# Lab 3.1 · Transformers vs. the Baseline
**Chapter 3 · Transformers and LLMs for Sentiment, Emotion and Stance**

**Goal:** run pretrained models, fine-tune a small model and audit where transformers still fail.
**Data:** `outputs/posts_ch2.parquet` from Lab 2.1 — the **same** train/test split.
**Compute:** CPU is enough for the walkthrough; use a **Colab GPU** for Challenge 2.

📖 Interactive study guide: [Chapter 3](https://mehrdadjalali-ai.github.io/SocialMediaAnayltics/interactive/Chapter3_Transformers_LLMs.html)

> The first run downloads the models from Hugging Face (no API key needed).

In [ ]:
from pathlib import Path
import warnings; warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA, OUTD = ROOT / "data", ROOT / "outputs"
OUTD.mkdir(exist_ok=True)
SEED = 42
import pandas as pd
from sklearn.metrics import f1_score
p = OUTD / "posts_ch2.parquet"
assert p.exists(), "Run Lab 2.1 first — it creates outputs/posts_ch2.parquet"
df = pd.read_parquet(p)
te = df[df["split"] == "test"].copy()
print(len(te), "test posts")

## Walkthrough · pretrained sentiment and emotion

In [ ]:
from transformers import pipeline
sent = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment-latest")
emo = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base")
texts = te["text_transformer"].tolist()
s_out = sent(texts, batch_size=32, truncation=True)
te["rob"] = [o["label"].lower() for o in s_out]
te["rob_score"] = [o["score"] for o in s_out]
te["emo"] = [o["label"] for o in emo(texts, batch_size=32, truncation=True)]
for col in ["vader", "lr", "rob"]:
    print(col, round(f1_score(te["sentiment"], te[col], average="macro"), 3))

### Zero-shot stance (NLI)

In [ ]:
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
post = "I am worried about electric cars, but I fully support the new subsidy policy."
zs(post, candidate_labels=["in favour of", "against", "no position on"],
   hypothesis_template="The author is {} the subsidy policy.")

## Challenge 1 · Benchmark

Compare pretrained RoBERTa, VADER and LR on the same test posts (macro-F1, per-class F1). Find **three posts** RoBERTa gets right that LR missed. Report per-language macro-F1 (en / de).

> 💡 **Hint:** Use `te[(te.rob == te.sentiment) & (te.lr != te.sentiment)]` and `te.groupby('lang')`.

In [ ]:
# Challenge 1 — your code here


**Your interpretation (Challenge 1):** _write 2–3 sentences — what did you find, and what could explain it?_

## Challenge 2 · Fine-tuning (Colab GPU)

Fine-tune `cardiffnlp/twitter-roberta-base-sentiment-latest` on the **training split** (2–4 epochs, lr 2e-5, seed 42). Choose the epochs on a validation split carved from training data — never on the test set. Report macro-F1 and training time.

> 💡 **Hint:** Use `AutoTokenizer`, `AutoModelForSequenceClassification(num_labels=3)` and `Trainer`. Map labels to ids in the model's label order.

In [ ]:
# Challenge 2 — your code here (run on a GPU)


**Your interpretation (Challenge 2):** _write 2–3 sentences — what did you find, and what could explain it?_

## Challenge 3 · Bias and failure audit

Collect **10 doubtful or confidently wrong** predictions and group them by cause: sarcasm, topic shortcut, dialect, identity terms, unclear label, time drift, other language.

> 💡 **Hint:** `te[te.rob != te.sentiment].sort_values('rob_score', ascending=False).head(10)`

In [ ]:
# Challenge 3 — your code here


**Your interpretation (Challenge 3):** _write 2–3 sentences — what did you find, and what could explain it?_

## Save for Chapter 4

Disclose model names, versions, access date and any LLM use in your worksheet.

In [ ]:
te.to_parquet(OUTD / "posts_ch3.parquet", index=False)
print("saved outputs/posts_ch3.parquet")